## Feature Engineering
---
Input  : `data_cleaned.csv` (output Notebook 1)  
Output : `data_engineered.csv` + `X_train.csv`, `X_test.csv`, `y_train.csv`, `y_test.csv`

### 2.1 Import Library

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import mutual_info_classif
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12
print('✅ Library berhasil diimport')

✅ Library berhasil diimport


### 2.2 Load Data Cleaned

In [2]:
df = pd.read_csv('data_cleaned.csv')
print(f'Shape data: {df.shape}')
df.head(5)

FileNotFoundError: [Errno 2] No such file or directory: 'data_cleaned.csv'

### 2.3 Analisis Korelasi Pearson

In [ ]:
num_cols = ['App Usage Time (min/day)', 'Screen On Time (hours/day)',
            'Battery Drain (mAh/day)', 'Number of Apps Installed',
            'Data Usage (MB/day)', 'Age', 'Gender', 'Operating System',
            'User Behavior Class']

corr_matrix = df[num_cols].corr(method='pearson')

plt.figure(figsize=(12, 9))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            mask=mask, square=True, linewidths=0.5,
            cbar_kws={'shrink': 0.8},
            annot_kws={'size': 10})
plt.title('Pearson Correlation Matrix – Fitur Numerik', fontsize=14, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('02_correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

# Korelasi terhadap target
target_corr = corr_matrix['User Behavior Class'].drop('User Behavior Class').sort_values(ascending=False)
print('\n=== Korelasi Fitur terhadap Target (User Behavior Class) ===')
print(target_corr.to_string())

### 2.4 Feature Engineering — Buat Fitur Baru

In [ ]:
df_eng = df.copy()

# 1. Rasio penggunaan baterai per menit penggunaan aplikasi
df_eng['Battery_per_App_Usage'] = (
    df_eng['Battery Drain (mAh/day)'] / (df_eng['App Usage Time (min/day)'] + 1)
).round(4)

# 2. Rasio data per menit penggunaan aplikasi
df_eng['Data_per_App_Usage'] = (
    df_eng['Data Usage (MB/day)'] / (df_eng['App Usage Time (min/day)'] + 1)
).round(4)

# 3. Efisiensi layar: menit penggunaan app dibagi jam layar menyala
df_eng['App_Screen_Ratio'] = (
    df_eng['App Usage Time (min/day)'] / (df_eng['Screen On Time (hours/day)'] + 0.01)
).round(4)

# 4. Interaksi: App Usage * Jumlah App (intensitas penggunaan)
df_eng['Usage_Intensity'] = (
    df_eng['App Usage Time (min/day)'] * df_eng['Number of Apps Installed']
)

# 5. Kategori usia (Age Group)
df_eng['Age_Group'] = pd.cut(
    df_eng['Age'],
    bins=[0, 24, 34, 44, 60],
    labels=[0, 1, 2, 3]  # Remaja-Dewasa Muda, Dewasa, Dewasa Pertengahan, Senior
).astype(int)

# 6. Total Digital Footprint Score (kombinasi linier fitur utama)
df_eng['Digital_Footprint'] = (
    df_eng['App Usage Time (min/day)'] / df_eng['App Usage Time (min/day)'].max() +
    df_eng['Battery Drain (mAh/day)'] / df_eng['Battery Drain (mAh/day)'].max() +
    df_eng['Data Usage (MB/day)'] / df_eng['Data Usage (MB/day)'].max()
).round(4)

new_features = ['Battery_per_App_Usage', 'Data_per_App_Usage', 'App_Screen_Ratio',
                'Usage_Intensity', 'Age_Group', 'Digital_Footprint']

print('✅ Fitur baru berhasil dibuat:')
for f in new_features:
    print(f'   - {f}')
print(f'\nShape setelah feature engineering: {df_eng.shape}')
df_eng[new_features].describe().round(3)

### 2.5 Mutual Information – Seleksi Fitur Terbaik

In [ ]:
X_all = df_eng.drop(columns=['User Behavior Class'])
y     = df_eng['User Behavior Class']

mi_scores = mutual_info_classif(X_all, y, random_state=42)
mi_df = pd.DataFrame({'Fitur': X_all.columns, 'MI Score': mi_scores})
mi_df = mi_df.sort_values('MI Score', ascending=False).reset_index(drop=True)

print('=== Mutual Information Score (Tingkat Kepentingan Fitur) ===')
print(mi_df.to_string(index=False))

plt.figure(figsize=(12, 7))
colors = ['#e74c3c' if i < 6 else '#95a5a6' for i in range(len(mi_df))]
bars = plt.barh(mi_df['Fitur'][::-1], mi_df['MI Score'][::-1], color=colors[::-1], edgecolor='black', linewidth=0.5)
plt.xlabel('Mutual Information Score')
plt.title('Seleksi Fitur – Mutual Information terhadap Target', fontweight='bold')
plt.axvline(x=mi_df['MI Score'].iloc[5], color='red', linestyle='--', alpha=0.7, label='Threshold (Top 6)')
plt.legend()
plt.tight_layout()
plt.savefig('02_mutual_information.png', dpi=150, bbox_inches='tight')
plt.show()

### 2.6 Pilih Fitur Final

In [ ]:
# Pilih semua fitur yang relevan (MI score > 0.01)
selected_features = mi_df[mi_df['MI Score'] > 0.01]['Fitur'].tolist()
print(f'Fitur terpilih ({len(selected_features)} fitur):')
for f in selected_features:
    print(f'  - {f}')

X = df_eng[selected_features]
y = df_eng['User Behavior Class']

### 2.7 Normalisasi Fitur (MinMaxScaler)

In [ ]:
scaler = MinMaxScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

print('✅ Normalisasi MinMax selesai')
print('Statistik setelah normalisasi:')
X_scaled.describe().round(3)

### 2.8 Split Data Train & Test

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

print('=== Pembagian Data ===')
print(f'Total data  : {len(X_scaled)}')
print(f'Data train  : {len(X_train)} ({len(X_train)/len(X_scaled)*100:.0f}%)')
print(f'Data test   : {len(X_test)}  ({len(X_test)/len(X_scaled)*100:.0f}%)')

# Distribusi kelas pada train & test
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
class_labels = {1:'Sangat\nRendah', 2:'Rendah', 3:'Sedang', 4:'Tinggi', 5:'Sangat\nTinggi'}
colors = ['#2ecc71','#3498db','#f39c12','#e74c3c','#8e44ad']

for ax, data, title in zip(axes, [y_train, y_test], ['Train Set', 'Test Set']):
    counts = data.value_counts().sort_index()
    ax.bar([class_labels[i] for i in counts.index], counts.values, color=colors, edgecolor='black')
    ax.set_title(f'Distribusi Kelas – {title}', fontweight='bold')
    ax.set_ylabel('Jumlah')
    for i, v in enumerate(counts.values):
        ax.text(i, v + 1, str(v), ha='center', fontweight='bold', fontsize=10)

plt.tight_layout()
plt.savefig('02_distribusi_split.png', dpi=150, bbox_inches='tight')
plt.show()

### 2.9 Simpan Data

In [ ]:
import joblib

df_eng.to_csv('data_engineered.csv', index=False)
X_train.to_csv('X_train.csv', index=False)
X_test.to_csv('X_test.csv', index=False)
y_train.to_csv('y_train.csv', index=False)
y_test.to_csv('y_test.csv', index=False)
joblib.dump(scaler, 'minmax_scaler.pkl')
joblib.dump(selected_features, 'selected_features.pkl')

print('✅ File tersimpan:')
print('   - data_engineered.csv')
print('   - X_train.csv, X_test.csv')
print('   - y_train.csv, y_test.csv')
print('   - minmax_scaler.pkl')
print('   - selected_features.pkl')